# Preliminary data processing

This script takes in the results from assembles-datasets and creates certain userful variables in the data.

In [1]:
import matplotlib.pyplot as plt
from mpl_toolkits.axes_grid1.inset_locator import inset_axes

import matplotlib as mpl
import matplotlib.ticker as ticker

import numpy as np
import pandas as pd
from pydmd import DMD
import os
import plotly.graph_objects as go
import plotly.express as px
import pickle

# import clustering packages
from sklearn.preprocessing import StandardScaler, RobustScaler, MinMaxScaler
from sklearn.decomposition import PCA
from sklearn.linear_model import LinearRegression
import seaborn as sns
from celluloid import Camera

plt.style.use('custom.mplstyle')
%config InlineBackend.figure_format = 'retina'
from tqdm import tqdm

from stoch_sim_model import *

## 0. Load data and build datasets

In [2]:
# Load data from infections
reg_model = ''
runs = '-1-'
comment = "sparse-reg" #"Nact-Ediv-vir" #"prim-Nact-Ediv-vir" #"full-reg-vir" #"act-reg-exp-reg" # mem-reg # comp_bias-Nact-Ediv-vir

d_mean = '/gscratch/scrubbed/oukogu/infoimmune/sim_output/no_cell_var/raw/stacked_data'+runs+'runs'+'-'+comment+'.pkl'
mean_df = pd.read_pickle(d_mean)

with pd.option_context('display.max_columns', None):
    display(mean_df)

,psi_Na_I,psi_Na_H,psi_Na_P,F0_Na,psi_NE_I,psi_NE_H,psi_NE_P,F0_NE,psi_EM_I,psi_EM_H,psi_EM_P,F0_EM,psi_EE_I,psi_EE_H,psi_EE_P,F0_EE,d_I,K_IE,b_I,K_EH,N_0,S_0,I_0,d_S,d_IE,b_H,d_H,max_Na,b_myc,d_myc,myc_thresh,t_bind,t_unbind,t_Na_div,t_E_div,t_M_div,t_E_die,t_act,p_load,T_max_pI,T_min_pI,harm_pI,harm_pS,max_pE,T_pE_max,T_pE_start,max_pM,T_pM_min,int_pP,int_pH,min_pS
0,-3.0,-3.0,-3.0,-4.0,0.0,0.0,0.0,-4.0,0.0,0.0,0.0,-4.0,0.0,0.0,0.0,-4.0,0.01,1.000000e+06,1.000000e-01,50000.0,300.0,10000000.0,1000000.0,0.01,16.0,1.0,2.0,4.0,4.0,1.0,1.0,1.0,0.75,0.25,0.333333,0.5,2.5,0.25,5.513457e+07,20.00,0.0,5.302757e+06,0.000000,300.0,20.00,20.000000,0.0,0.000000,0.000000,2.073697e+06,6.028307e+06
1,-3.0,-3.0,-3.0,-4.0,0.0,0.0,0.0,-4.0,0.0,0.0,0.0,-4.0,0.0,0.0,0.0,-4.0,0.01,1.000000e+06,1.000000e-01,50000.0,1000.0,10000000.0,1000000.0,0.01,16.0,1.0,2.0,4.0,4.0,1.0,1.0,1.0,0.75,0.25,0.333333,0.5,2.5,0.25,5.513457e+07,20.00,0.0,5.302757e+06,0.000000,1000.0,20.00,20.000000,0.0,0.000000,0.000000,2.073697e+06,6.028307e+06
2,-3.0,-3.0,-3.0,-4.0,0.0,0.0,0.0,-4.0,0.0,0.0,0.0,-4.0,0.0,0.0,0.0,-4.0,0.01,1.000000e+07,0.000000e+00,50000.0,300.0,10000000.0,0.0,0.01,16.0,1.0,2.0,4.0,4.0,1.0,1.0,1.0,0.75,0.25,0.333333,0.5,2.5,0.25,1.978648e+08,0.00,0.0,0.000000e+00,0.000000,300.0,20.00,20.000000,0.0,0.000000,0.000000,6.277409e+04,9.876119e+06
3,-3.0,-3.0,-3.0,-4.0,0.0,0.0,0.0,-4.0,0.0,0.0,0.0,-4.0,0.0,0.0,0.0,-4.0,0.01,1.000000e+07,1.000000e-01,50000.0,300.0,10000000.0,1000000.0,0.01,16.0,1.0,2.0,4.0,4.0,1.0,1.0,1.0,0.75,0.25,0.333333,0.5,2.5,0.25,5.513457e+07,20.00,0.0,5.302757e+06,0.000000,300.0,20.00,20.000000,0.0,0.000000,0.000000,2.073697e+06,6.028307e+06
4,-3.0,-3.0,-3.0,-4.0,0.0,0.0,0.0,-4.0,0.0,0.0,0.0,-4.0,0.0,0.0,0.0,-4.0,0.01,1.000000e+07,1.000000e-01,50000.0,1000.0,10000000.0,1000000.0,0.01,16.0,1.0,2.0,4.0,4.0,1.0,1.0,1.0,0.75,0.25,0.333333,0.5,2.5,0.25,5.513457e+07,20.00,0.0,5.302757e+06,0.000000,1000.0,20.00,20.000000,0.0,0.000000,0.000000,2.073697e+06,6.028307e+06
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
33547495,3.0,3.0,3.0,4.0,0.0,0.0,0.0,4.0,0.0,0.0,0.0,4.0,0.0,0.0,0.0,4.0,1.00,1.291550e+05,1.500000e-07,50000.0,300.0,10000000.0,1000.0,0.01,16.0,1.0,2.0,4.0,4.0,1.0,1.0,1.0,0.75,0.25,0.333333,0.5,2.5,0.25,3.878500e+06,17.36,0.0,4.454101e+06,20570.804342,49199.0,12.68,10.661700,43539.0,12.817163,101572.606434,1.834463e+06,5.772443e+06
33547496,3.0,3.0,3.0,4.0,0.0,0.0,0.0,4.0,0.0,0.0,0.0,4.0,0.0,0.0,0.0,4.0,1.00,2.154435e+05,1.500000e-07,50000.0,300.0,10000000.0,1000.0,0.01,16.0,1.0,2.0,4.0,4.0,1.0,1.0,1.0,0.75,0.25,0.333333,0.5,2.5,0.25,4.135715e+06,16.97,0.0,4.670295e+06,18833.291302,45902.0,13.51,10.341933,40585.0,12.518374,84764.278833,1.963819e+06,5.570095e+06
33547497,3.0,3.0,3.0,4.0,0.0,0.0,0.0,4.0,0.0,0.0,0.0,4.0,0.0,0.0,0.0,4.0,1.00,3.593814e+05,1.500000e-07,50000.0,300.0,10000000.0,1000.0,0.01,16.0,1.0,2.0,4.0,4.0,1.0,1.0,1.0,0.75,0.25,0.333333,0.5,2.5,0.25,4.382421e+06,16.20,0.0,4.869774e+06,16791.918824,43363.0,14.31,10.999967,38403.0,12.552595,72848.984882,2.092529e+06,5.384759e+06
33547498,3.0,3.0,3.0,4.0,0.0,0.0,0.0,4.0,0.0,0.0,0.0,4.0,0.0,0.0,0.0,4.0,1.00,5.994843e+05,1.500000e-07,50000.0,300.0,10000000.0,1000.0,0.01,16.0,1.0,2.0,4.0,4.0,1.0,1.0,1.0,0.75,0.25,0.333333,0.5,2.5,0.25,4.558860e+06,16.03,0.0,5.008150e+06,14020.788244,37553.0,14.78,11.407433,33289.0,12.000301,52858.872028,2.179913e+06,5.253647e+06


## 1. Understanding the statistics of responses to an infection

In [3]:
# Create additional variables
virs = np.unique(mean_df[['I_0','d_I','K_IE','b_I','K_EH','N_0']].to_numpy(), axis = 0)

mean_df['antigenicity_over_harm'] = antigenicity_over_harm(mean_df)
mean_df['T_pE_clear'] = mean_df['T_pE_max'] - mean_df['T_max_pI']
mean_df['max_pM_fold'] = np.log(1 + mean_df['max_pM']/mean_df['N_0'])
mean_df['stim_pI'] = np.log(1 + (mean_df['p_load']/mean_df['K_IE'])/(mean_df['T_min_pI'] + sim_duration*(mean_df['T_min_pI'] == 0)))
mean_df['stim_pH'] = np.log(1 + (mean_df['int_pH']/mean_df['K_EH'])/(mean_df['T_min_pI'] + sim_duration*(mean_df['T_min_pI'] == 0)))
mean_df['stim_pP'] = np.log(1 + (mean_df['int_pP']/mean_df['K_EH'])/((mean_df['T_pE_max'] - mean_df['T_pE_start'])*((mean_df['T_pE_max'] - mean_df['T_pE_start']) > 0) + sim_duration*((mean_df['T_pE_max'] - mean_df['T_pE_start']) <= 0)))
# ((mean_df['T_pE_max'] - mean_df['T_pE_start'])*(mean_df['T_pE_max'] > mean_df['T_pE_start']) + sim_duration*(mean_df['T_pE_max'] <= mean_df['T_pE_start']))
mean_df['max_pE_fold'] = np.log(1 + mean_df['max_pE']/mean_df['N_0'])

# identify Biologically evidenced networks
keep_vars = ['harm_pI','harm_pS', 'max_pM_fold', 'max_pE_fold', 'T_pM_min', 
             'T_min_pI', 'T_max_pI', 'T_pE_start', 'T_pE_clear',
             'stim_pI', 'stim_pH', 'stim_pP',
             'min_pS', 'antigenicity_over_harm']

In [7]:
# save data sets
infection_scenarios = []
no_eff_data = [[] for i in np.arange(len(virs))]
b_S = d_S*S_0

for l, (I_0, d_I, K_IE, b_I, K_EH, N_0) in enumerate(tqdm(virs)):
    data = mean_df.loc[(mean_df["d_I"] == d_I)*(mean_df["K_IE"] == K_IE)*(mean_df["b_I"] == b_I)*(mean_df["K_EH"] == K_EH)*(mean_df["N_0"] == N_0)*(mean_df["I_0"] == I_0), 
    ['b_I','d_I', 'K_IE', 'I_0','S_0', 'N_0', 'd_S', 'K_EH'] + Na_reg + NE_reg + EM_reg + EE_reg + keep_vars]

    # compute infection harm without T cell response
    no_eff_data[l] = lin_stoch_sim(N_0 = 0, I_0 = I_0, K_IE = K_IE, d_I = d_I, b_I = b_I) #, vir_model = "indep_harm" if b_I > 0 and d_I > 10*d_S else "autoimmune")
    no_eff_stats = no_eff_data[l]["summary_stats"]

    data.loc[:,"harm_pI_noprotection"] = no_eff_stats[3]/(b_S*sim_duration)
    data.loc[:,"peff_protection"] = np.maximum(0.0, (no_eff_stats[3] - data['harm_pI'].to_numpy())/(b_S*(data['T_min_pI'] + sim_duration*(data['T_min_pI'] == 0))))
    data.loc[:,"peff_toxicity"] = data['harm_pS'].to_numpy()/(b_S*(data['T_min_pI'] + sim_duration*(data['T_min_pI'] == 0)))

    infection_scenarios.append(data)

# stack datasets
clustered_mean_df = pd.concat(infection_scenarios)
clustered_mean_df.to_pickle('/gscratch/scrubbed/oukogu/infoimmune/sim_output/no_cell_var/summary_stats/mean/processed_data'+runs+'runs'+'-'+comment+'.pkl')

with open('/gscratch/scrubbed/oukogu/infoimmune/sim_output/no_cell_var/summary_stats/mean/list_processed_data'+runs+'runs'+'-'+comment+'.pkl', 'wb') as f:
    pickle.dump(infection_scenarios, f)

 96%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████▍    | 104/108 [03:14<00:07,  1.89s/it]/mmfs1/home/oukogu/infoimmune/stoch_sim_model.py:98: RuntimeWarning: invalid value encountered in log
  psi_1 * np.log(1 + sig_1 / K_1)
/mmfs1/home/oukogu/infoimmune/stoch_sim_model.py:99: RuntimeWarning: invalid value encountered in log
  + psi_2 * np.log(1 + sig_2 / K_2)
 97%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████▌   | 105/108 [03:16<00:05,  1.88s/it]/mmfs1/home/oukogu/infoimmune/stoch_sim_model.py:98: RuntimeWarning: invalid value encountered in log
  psi_1 * np.log(1 + sig_1 / K_1)
/mmfs1/home/oukogu/infoimmune/stoch_sim_model.py:99: RuntimeWarning: invalid value encountered in log
  + psi_2 * np.log(1 + sig_2 / K_2)
 98%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████

In [8]:
del clustered_mean_df, infection_scenarios